In [1]:
# ================================================================
# NOTEBOOK : nb_silver_sales
# Read from  : bronze_lakehouse → SalesTransactions
# Write to   : silver_lakehouse → silver_sales  (default lakehouse)
# ================================================================


StatementMeta(, 9e61550b-2b9e-4d73-9e56-d70a98f7ac36, 3, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, round, trim, initcap, upper


# ── READ from bronze (use your exact table name from SSMS migration) ──
bronze_df = spark.sql("SELECT * FROM bronze_lakehouse.bronze_SalesTrans")
print(f"[BRONZE]:{bronze_df.count()} rows")

display(bronze_df)

StatementMeta(, 9e61550b-2b9e-4d73-9e56-d70a98f7ac36, 7, Finished, Available, Finished, False)

[BRONZE]:2000 rows


SynapseWidget(Synapse.DataFrame, fc684707-eabe-4e8e-a093-3db124e44d1c)

In [ ]:
bronze_df.printSchema()

StatementMeta(, 285ded74-7802-4ac9-9d16-0d810544a32c, 14, Finished, Available, Finished, False)

root
 |-- TransactionID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- DiscountAmount: decimal(10,2) (nullable = true)
 |-- TotalAmount: decimal(10,2) (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- CreatedAt: timestamp (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [14]:
# ── STEP 1: Deduplication ─────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, trim, upper, round
from pyspark.sql.window import Window

# Step 1: Define window (group + sort)
window = Window.partitionBy("TransactionID").orderBy(F.desc("LastModifiedDate"))

# Step 2: Add row_number column
deduped_df  = (bronze_df.withColumn("_row_num", F.row_number().over(window))

# Step 3: Keep only latest records (rn = 1)
.filter(F.col("_row_num") == 1)

# Step 4: Drop helper column
.drop("_row_num")

)

print(f"[DEDUP] After dedup:{deduped_df.count()}")



# ── STEP 2: Remove nulls in critical columns ──────────────────────────

clean_df = (deduped_df.filter(col("TransactionID").isNotNull())
.filter(col("StoreID").isNotNull())
.filter(col("ProductID").isNotNull())
.filter(col("TotalAmount").isNotNull())
.filter(col("TransactionDate").isNotNull())
)

print(f"[CLEAN] :{clean_df.count()} ")
# ── STEP 3: Data quality — flag amount mismatches ─────────────────────

flagged_df = clean_df.withColumn("_dq_flag",
when(F.abs(col("TotalAmount").cast("float")) > 500000, "AMOUNT_OUTLIER") # suspiciously large
.when(col("Quantity") == 0, "ZERO_QUANTITY")
.otherwise("PASS")
)


# Quarantine bad records (save to bronze_lakehouse for ops review)
quarantine_df = flagged_df.filter(col("_dq_flag") != "PASS")
if quarantine_df.count() > 0:
    quarantine_df.write.format("delta").mode("append")\
    .saveAsTable("bronze_lakehouse.quarantine_sales")
print(f"[WARN] Quarantined rows :{quarantine_df.count()}")

clean_df = flagged_df.filter(col("_dq_flag") == "PASS").drop("_dq_flag")

# ── STEP 4: Fix data types ────────────────────────────────────────────

typed_df = ( clean_df.withColumn("TransactionDate",    col("TransactionDate").cast("date"))
.withColumn("TotalAmount",  col("TotalAmount").cast("decimal(18,2)"))
.withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)"))
.withColumn("DiscountAmount",    col("DiscountAmount").cast("decimal(10,2)"))
.withColumn("Quantity",          col("Quantity").cast("integer"))
.withColumn("CreatedAt",         col("CreatedAt").cast("timestamp"))
.withColumn("LastModifiedDate",     col("LastModifiedDate").cast("timestamp"))
)

# ── STEP 5: Standardise strings ───────────────────────────────────────
typed_df = ( typed_df.withColumn("PaymentMethod", upper(trim(col("PaymentMethod"))))
.withColumn("ProductID",    upper(trim(col("ProductID"))))
.withColumn("StoreID",   upper(trim(col("StoreID"))))
)

# ── STEP 6: Derived / enrichment columns ─────────────────────────────

silver_df = ( typed_df.withColumn("Year",    F.year("TransactionDate"))
.withColumn("Month",        F.month("TransactionDate"))
.withColumn("Quarter",      F.quarter("TransactionDate"))
.withColumn("WeekOfYear",     F.weekofyear("TransactionDate"))
.withColumn("DayOfWeek",   F.dayofweek("TransactionDate"))

# 1=Sunday,7=Saturday in Spark
.withColumn("IsWeekend",     when(col("DayOfWeek").isin([1,7]),True).otherwise(False))
.withColumn("IsReturn",     when(col("TotalAmount") <0, True).otherwise(False))

# NetRevenue: returns contribute 0 to revenue, not negative
.withColumn("NetRevenue",       when(col("IsReturn"), F.lit(0.0)).otherwise(col("TotalAmount").cast("double")))

# Effective unit price after discount
.withColumn("EffectiveUnitPrice",
                round((col("TotalAmount") / col("Quantity")), 2))

#Metadata
.withColumn("_silver_load_ts", F.current_timestamp())
)

# ── WRITE to silver_lakehouse ─────────────────────────────────────────
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")\
.saveAsTable("silver_sales")


print(f"[DONE]silver_sales written:{silver_df.count()} rows")

display(silver_df)


StatementMeta(, 22b91917-ffb4-436b-a28e-ed019f685788, 16, Finished, Available, Finished, False)

[DEDUP] After dedup:2000
[CLEAN] :2000 
[WARN] Quarantined rows :0
[DONE]silver_sales written:2000rows


SynapseWidget(Synapse.DataFrame, 8f3ea850-b182-4ffd-a24e-fdefa1eade1b)

In [5]:
weather_df = spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_weather")
display(weather_df)


StatementMeta(, 327ea440-0630-4432-8c28-510809a4a2bf, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0abc966-4251-4786-a60f-339e9926b62a)